# NanoForge Studio on a Colab L4

Choose **Runtime → Change runtime type → L4 GPU**, then run the cell below. It installs the backend and frontend, verifies CUDA, and opens the full NanoForge UI **below the cell** through Colab's notebook proxy.

**All model and training settings are in the UI.** Choose Mini or Custom, tokenizer, parameter budget, learning rate, batch size, steps, precision and TF32. Use **Data & Mixer** to import datasets and set weights. Click **New experiment** when changing architecture, then **Start training**. The launcher leaves the trainer idle and imports no datasets automatically. If Dolly already exists it is selected; otherwise the included starter sample is selected.

Use **Download .pt checkpoint** to export saved weights. `/content` is temporary, so preserve checkpoints before ending the runtime. You can select/upload previous checkpoints from the same UI.

Stop an earlier headless training cell before launching this Studio. Leave the Colab runtime connected while using it. No public tunnel, tunnel account or API key is used. The UI uses HTTP polling for progress through the notebook proxy.


In [ ]:
from pathlib import Path
import subprocess

# Stop only the Studio process started by an earlier execution of this cell.
if 'nanoforge_studio' in globals():
    nanoforge_studio.stop()

REPO = Path('/content/nanoforge')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/bananaosint/nanoforge.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)

startup = {'__name__': 'nanoforge_colab_ui'}
script = REPO/'scripts/colab_ui.py'
exec(compile(script.read_text(),str(script),'exec'),startup)
nanoforge_studio = startup['launch_studio'](repo=REPO)
startup['show_studio'](nanoforge_studio)


## Session controls

The UI runs in your Colab runtime and controls its L4 GPU. The app shows **COLAB STUDIO**, a Colab connection label and the detected device. Local desktop Studio remains separate.

For server diagnostics, run `print(nanoforge_studio.logs())` in another cell. To stop the server safely and save the active training step/checkpoint, run `nanoforge_studio.stop()`. Rerunning the launcher cell does this first; it does not kill unrelated processes. If shutdown needs more time, wait for the checkpoint and call `stop()` again.

If a large download is blocked by the proxy/browser, use Colab's Files panel under `/content/nanoforge-l4-run/checkpoints/`. Copy a `.pt` back into local Studio using **Import .pt checkpoint**. For Drive persistence, mount Drive first and pass an existing Drive folder as `home=` to `launch_studio`; model settings still stay in the UI.

The original cell that starts training immediately is retained in `NanoForge_L4_Headless.ipynb`.
